# Refresher: Linear Algebra & Calculus

## 0. Step-by-Step Worked Example — Start Here (Beginner Friendly)

> 🧑‍🎓 **New to this topic? Start here.** This is a gentle, fully runnable walkthrough that
> builds up the core idea one tiny step at a time. Each step **prints** the numbers it
> computes and **draws a picture** so you can *see* what is happening. Run the cells in order
> from top to bottom. Nothing here needs the internet or any downloaded data.

### The Big Picture — What You'll Learn

- A **dot product** and **norm** summarize vectors; a **matrix** transforms them.
- The **gradient** points uphill; gradient descent steps the opposite way.
- All of it reduces to small, checkable arithmetic.

### Step 0 — Set up our tools

We import NumPy (arrays + math) and Matplotlib (pictures), fix a **seed** for reproducibility,
and define a tiny `log()` helper so every printed line is clearly labeled.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.figsize"] = (6, 4)

def log(label, value):
    print(f"[{label}] {value}")

log("setup", "tools ready — NumPy + Matplotlib imported, seed fixed to 0")

▶ What you'll see: one line confirming the tools are ready.

### Step 1 — Vectors and a matrix transform

The **dot product** multiplies matching components and adds; the **norm** is the Pythagorean length; a **matrix times a vector** produces a new vector (each output = a row dotted with the vector).

In [ ]:
v = np.array([3, 4]); M = np.array([[1, 2], [3, 4]])
log("dot(v, v)", int(np.dot(v, v)))
log("norm(v) = sqrt(dot(v,v))", round(np.linalg.norm(v), 3))
Mv = M @ v
log("M v (each row dotted with v)", Mv.tolist())
assert round(np.linalg.norm(v), 3) == 5.0

▶ What you'll see: dot 25, norm 5.0, and `M v = [11, 25]`.

### Step 2 — Gradient and one descent step

For `f(x,y) = x² + y²`, the **gradient** is `[2x, 2y]` — it points uphill. **Gradient descent** takes a small step in the *opposite* direction to go downhill toward the minimum at the origin.

In [ ]:
grad = lambda p: 2 * np.array(p, float)                       # gradient of x^2 + y^2
p = np.array([3.0, 4.0]); log("gradient at (3,4)", grad(p).tolist())
assert (grad(p) == [6, 8]).all()
lr = 0.1; p_new = p - lr * grad(p)                            # step downhill
log("after one step p - 0.1*grad", np.round(p_new, 2).tolist())

xs = np.linspace(-4, 4, 40); ys = np.linspace(-4, 4, 40); XX, YY = np.meshgrid(xs, ys)
plt.contour(XX, YY, XX**2 + YY**2, levels=12)
plt.annotate("", xy=p_new, xytext=p, arrowprops=dict(arrowstyle="->", color="red"))
plt.title("gradient descent step on x^2+y^2"); plt.scatter([0], [0], marker="*", s=200); plt.show()

▶ What you'll see: gradient [6, 8], a step to [2.4, 3.2], and an arrow moving toward the center.

## 1. Overview

Linear algebra gives machine learning its language for data tables, parameter vectors, projections, curvature, and directions of variation. Calculus tells us how scalar objectives change when vectors or matrices change. The working intuition is that most ML formulas become manageable once we track shapes, multiply carefully, measure size with norms, identify eigen-directions, and compute gradients and Hessians by hand.

## 2. Key Idea

### Vectors, matrices, and special matrices

A vector $x\in\mathbb{R}^n$ is

$$
x=\begin{bmatrix}x_1\\x_2\\\vdots\\x_n\end{bmatrix}\in\mathbb{R}^n.
$$

Read it as an $n\times1$ column-vector.

A matrix $A\in\mathbb{R}^{m\times n}$ is

$$
A=\begin{bmatrix}
A_{1,1} & \cdots & A_{1,n}\\
\vdots & & \vdots\\
A_{m,1} & \cdots & A_{m,n}
\end{bmatrix}\in\mathbb{R}^{m\times n}.
$$

Read the shape as rows by columns, with $A_{i,j}$ in row $i$ and column $j$.

The identity matrix is

$$
I=\begin{bmatrix}
1 & 0 & \cdots & 0\\
0 & \ddots & \ddots & \vdots\\
\vdots & \ddots & \ddots & 0\\
0 & \cdots & 0 & 1
\end{bmatrix}.
$$

Read it as the matrix satisfying $A\times I=I\times A=A$ whenever the products are defined.

A diagonal matrix is

$$
D=\begin{bmatrix}
d_1 & 0 & \cdots & 0\\
0 & \ddots & \ddots & \vdots\\
\vdots & \ddots & \ddots & 0\\
0 & \cdots & 0 & d_n
\end{bmatrix}=\operatorname{diag}(d_1,\ldots,d_n).
$$

Read it as coordinate-wise scaling.

### Matrix operations

For $x,y\in\mathbb{R}^n$, the inner product is

$$
x^T y=\sum_{i=1}^n x_i y_i\in\mathbb{R}.
$$

Read it as multiply matching entries and sum.

For $x\in\mathbb{R}^m$ and $y\in\mathbb{R}^n$, the outer product is

$$
xy^T=\begin{bmatrix}
x_1y_1 & \cdots & x_1y_n\\
\vdots & & \vdots\\
x_my_1 & \cdots & x_my_n
\end{bmatrix}\in\mathbb{R}^{m\times n}.
$$

Read it as all pairwise coordinate products.

For $A\in\mathbb{R}^{m\times n}$ and $x\in\mathbb{R}^n$,

$$
Ax=\begin{bmatrix}a_{r,1}^T x\\\vdots\\a_{r,m}^T x\end{bmatrix}=\sum_{i=1}^n a_{c,i}x_i\in\mathbb{R}^m.
$$

Read it either as row inner products or as a weighted sum of columns.

For $A\in\mathbb{R}^{m\times n}$ and $B\in\mathbb{R}^{n\times p}$,

$$
AB=\begin{bmatrix}
a_{r,1}^T b_{c,1} & \cdots & a_{r,1}^T b_{c,p}\\
\vdots & & \vdots\\
a_{r,m}^T b_{c,1} & \cdots & a_{r,m}^T b_{c,p}
\end{bmatrix}=\sum_{i=1}^n a_{c,i}b_{r,i}^T\in\mathbb{R}^{n\times p}.
$$

Read each entry as a row of $A$ dotted with a column of $B$, and the sum as adding outer products.

The transpose satisfies

$$
\forall i,j,\qquad A^T_{i,j}=A_{j,i}.
$$

Read it as swapping rows and columns; for products, $(AB)^T=B^TA^T$.

The inverse of an invertible square matrix satisfies

$$
AA^{-1}=A^{-1}A=I.
$$

Read it as the operation that undoes $A$; for products, $(AB)^{-1}=B^{-1}A^{-1}$.

The trace is

$$
\operatorname{tr}(A)=\sum_{i=1}^n A_{i,i}.
$$

Read it as the diagonal sum; $\operatorname{tr}(A^T)=\operatorname{tr}(A)$ and $\operatorname{tr}(AB)=\operatorname{tr}(BA)$.

The determinant is

$$
\det(A)=|A|=\sum_{j=1}^n (-1)^{i+j}A_{i,j}|A_{\backslash i,\backslash j}|.
$$

Read it as signed volume scaling; $A$ is invertible if and only if $|A|\neq0$, with $|AB|=|A||B|$ and $|A^T|=|A|$.

### Matrix properties

A matrix decomposes into symmetric and antisymmetric parts:

$$
A=\underbrace{\frac{A+A^T}{2}}_{\textrm{Symmetric}}+\underbrace{\frac{A-A^T}{2}}_{\textrm{Antisymmetric}}.
$$

Read this as splitting the transpose-stable part from the transpose-sign-flipping part.

A norm $N:V\longrightarrow[0,+\infty[$ satisfies

$$
N(x+y)\leq N(x)+N(y),\qquad N(ax)=|a|N(x),\qquad N(x)=0\Rightarrow x=0.
$$

Read a norm as a valid length function.

The main norms are

$$
\lVert x\rVert_1=\sum_{i=1}^n |x_i|,
\qquad
\lVert x\rVert_2=\sqrt{\sum_{i=1}^n x_i^2},
$$

$$
\lVert x\rVert_p=\left(\sum_{i=1}^n x_i^p\right)^{\frac{1}{p}},
\qquad
\lVert x\rVert_\infty=\max_i |x_i|.
$$

Read them as absolute-size, Euclidean-size, reference $p$-size, and maximum-coordinate size.

A set of vectors is linearly dependent if one vector can be written as a linear combination of the others. Read dependence as redundancy.

The rank $\operatorname{rank}(A)$ is the dimension of the vector space generated by the columns of $A$. Read rank as the number of independent column directions.

A matrix is positive semi-definite when

$$
A=A^T \qquad \textrm{and}\qquad \forall x\in\mathbb{R}^n,\quad x^TAx\geq0.
$$

Read PSD as nonnegative quadratic-form curvature; $A\succ0$ makes the inequality strict for all nonzero $x$.

An eigenvalue-eigenvector pair satisfies

$$
Az=\lambda z.
$$

Read $z$ as a direction kept fixed by $A$ except for scaling by $\lambda$.

For symmetric $A\in\mathbb{R}^{n\times n}$, the spectral theorem gives

$$
\exists \Lambda \textrm{ diagonal},\quad A=U\Lambda U^T,
$$

where $U$ is real orthogonal and $\Lambda=\operatorname{diag}(\lambda_1,\ldots,\lambda_n)$. Read this as an orthogonal eigen-direction decomposition.

The singular-value decomposition says

$$
A=U\Sigma V^T,
$$

with $U$ $m\times m$ unitary, $\Sigma$ $m\times n$ diagonal, and $V$ $n\times n$ unitary. Read it as a universal stretch-rotate factorization.

### Matrix calculus

For scalar $f:\mathbb{R}^{m\times n}\to\mathbb{R}$,

$$
(\nabla_A f(A))_{i,j}=\frac{\partial f(A)}{\partial A_{i,j}}.
$$

Read the matrix gradient as the same shape as $A$, filled with entrywise partial derivatives.

For scalar $f:\mathbb{R}^n\to\mathbb{R}$,

$$
(\nabla_x^2 f(x))_{i,j}=\frac{\partial^2 f(x)}{\partial x_i\partial x_j}.
$$

Read the Hessian as the matrix of second derivatives and local curvature.

The reference matrix-calculus identities are

$$
\nabla_A\operatorname{tr}(AB)=B^T,
$$

$$
\nabla_{A^T}f(A)=(\nabla_A f(A))^T,
$$

$$
\nabla_A\operatorname{tr}(ABA^TC)=CAB+C^TAB^T,
$$

$$
\nabla_A |A|=|A|(A^{-1})^T.
$$

Read them as trace-linear, transpose-gradient, two-appearance quadratic trace, and determinant-sensitivity rules. Dividing the determinant identity by $|A|$ gives

$$
\nabla_A\log|A|=(A^{-1})^T.
$$

## 3. Worked Examples

### 🟡 Easy

#### E1. Matrix-vector multiplication by rows and columns

**Problem.** Let

$$
A=\begin{bmatrix}2&-1&0\\1&3&4\end{bmatrix},
\qquad
x=\begin{bmatrix}5\\2\\-1\end{bmatrix}.
$$

Compute $Ax$ two ways and check the shape.

**Solution.**

The shapes are $A\in\mathbb{R}^{2\times3}$ and $x\in\mathbb{R}^3$, so $Ax\in\mathbb{R}^2$.

By row inner products,

$$
Ax=\begin{bmatrix}
2(5)+(-1)(2)+0(-1)\\
1(5)+3(2)+4(-1)
\end{bmatrix}.
$$

Compute each entry:

$$
2(5)+(-1)(2)+0(-1)=10-2+0=8,
$$

$$
1(5)+3(2)+4(-1)=5+6-4=7.
$$

Thus

$$
Ax=\begin{bmatrix}8\\7\end{bmatrix}.
$$

By weighted columns,

$$
Ax=5\begin{bmatrix}2\\1\end{bmatrix}
+2\begin{bmatrix}-1\\3\end{bmatrix}
+(-1)\begin{bmatrix}0\\4\end{bmatrix}.
$$

Scale and add:

$$
\begin{bmatrix}10\\5\end{bmatrix}
+\begin{bmatrix}-2\\6\end{bmatrix}
+\begin{bmatrix}0\\-4\end{bmatrix}
=\begin{bmatrix}8\\7\end{bmatrix}.
$$

Therefore

$$
\boxed{Ax=\begin{bmatrix}8\\7\end{bmatrix}\in\mathbb{R}^2.}
$$

#### E2. Inner product, outer product, and norms

**Problem.** Let

$$
u=\begin{bmatrix}1\\-2\\2\end{bmatrix},
\qquad
v=\begin{bmatrix}3\\0\\-1\end{bmatrix}.
$$

Compute $u^Tv$, $uv^T$, $\lVert u\rVert_1$, $\lVert u\rVert_2$, and $\lVert u\rVert_\infty$.

**Solution.**

The inner product is

$$
u^Tv=1(3)+(-2)(0)+2(-1)=3+0-2=1.
$$

The outer product is

$$
uv^T=\begin{bmatrix}1\\-2\\2\end{bmatrix}\begin{bmatrix}3&0&-1\end{bmatrix}.
$$

Scale $v^T$ by each entry of $u$:

$$
1\begin{bmatrix}3&0&-1\end{bmatrix}=\begin{bmatrix}3&0&-1\end{bmatrix},
$$

$$
(-2)\begin{bmatrix}3&0&-1\end{bmatrix}=\begin{bmatrix}-6&0&2\end{bmatrix},
$$

$$
2\begin{bmatrix}3&0&-1\end{bmatrix}=\begin{bmatrix}6&0&-2\end{bmatrix}.
$$

So

$$
uv^T=\begin{bmatrix}3&0&-1\\-6&0&2\\6&0&-2\end{bmatrix}.
$$

The norms are

$$
\lVert u\rVert_1=|1|+|-2|+|2|=1+2+2=5,
$$

$$
\lVert u\rVert_2=\sqrt{1^2+(-2)^2+2^2}=\sqrt{1+4+4}=\sqrt9=3,
$$

$$
\lVert u\rVert_\infty=\max\{|1|,|-2|,|2|\}=\max\{1,2,2\}=2.
$$

Therefore

$$
\boxed{u^Tv=1,\quad
uv^T=\begin{bmatrix}3&0&-1\\-6&0&2\\6&0&-2\end{bmatrix},\quad
\lVert u\rVert_1=5,\quad
\lVert u\rVert_2=3,\quad
\lVert u\rVert_\infty=2.}
$$

#### E3. Determinant and invertibility of a $2\times2$ matrix

**Problem.** Let

$$
B=\begin{bmatrix}4&7\\2&6\end{bmatrix}.
$$

Compute the determinant, decide whether $B^{-1}$ exists, and verify the inverse.

**Solution.**

For $B=\begin{bmatrix}a&b\\c&d\end{bmatrix}$, the $2\times2$ determinant is $ad-bc$. Thus

$$
|B|=4(6)-7(2)=24-14=10.
$$

Since $10\neq0$, the inverse exists:

$$
B^{-1}=\frac{1}{10}\begin{bmatrix}6&-7\\-2&4\end{bmatrix}
=\begin{bmatrix}\frac35&-\frac7{10}\\-\frac15&\frac25\end{bmatrix}.
$$

Verify:

$$
BB^{-1}=\begin{bmatrix}4&7\\2&6\end{bmatrix}
\begin{bmatrix}\frac35&-\frac7{10}\\-\frac15&\frac25\end{bmatrix}.
$$

The four entries are

$$
4\left(\frac35\right)+7\left(-\frac15\right)=\frac{12}{5}-\frac75=1,
$$

$$
4\left(-\frac7{10}\right)+7\left(\frac25\right)=-\frac{28}{10}+\frac{14}{5}=-\frac{14}{5}+\frac{14}{5}=0,
$$

$$
2\left(\frac35\right)+6\left(-\frac15\right)=\frac65-\frac65=0,
$$

$$
2\left(-\frac7{10}\right)+6\left(\frac25\right)=-\frac75+\frac{12}{5}=1.
$$

Therefore $BB^{-1}=I$, and

$$
\boxed{|B|=10,\qquad B^{-1}=\frac1{10}\begin{bmatrix}6&-7\\-2&4\end{bmatrix}.}
$$

#### E4. Trace and transpose identities

**Problem.** Let

$$
C=\begin{bmatrix}1&2\\3&4\end{bmatrix},
\qquad
D=\begin{bmatrix}0&5\\-1&2\end{bmatrix}.
$$

Compute $CD$, $(CD)^T$, $D^TC^T$, $\operatorname{tr}(CD)$, and $\operatorname{tr}(DC)$.

**Solution.**

First,

$$
CD=\begin{bmatrix}1&2\\3&4\end{bmatrix}\begin{bmatrix}0&5\\-1&2\end{bmatrix}
=\begin{bmatrix}
1(0)+2(-1)&1(5)+2(2)\\
3(0)+4(-1)&3(5)+4(2)
\end{bmatrix}.
$$

Thus

$$
CD=\begin{bmatrix}-2&9\\-4&23\end{bmatrix},
\qquad
(CD)^T=\begin{bmatrix}-2&-4\\9&23\end{bmatrix}.
$$

Also,

$$
D^T=\begin{bmatrix}0&-1\\5&2\end{bmatrix},
\qquad
C^T=\begin{bmatrix}1&3\\2&4\end{bmatrix}.
$$

Then

$$
D^TC^T=\begin{bmatrix}0&-1\\5&2\end{bmatrix}\begin{bmatrix}1&3\\2&4\end{bmatrix}
=\begin{bmatrix}-2&-4\\9&23\end{bmatrix}.
$$

So $(CD)^T=D^TC^T$.

For traces,

$$
\operatorname{tr}(CD)=-2+23=21.
$$

The reverse product is

$$
DC=\begin{bmatrix}0&5\\-1&2\end{bmatrix}\begin{bmatrix}1&2\\3&4\end{bmatrix}
=\begin{bmatrix}15&20\\5&6\end{bmatrix}.
$$

Thus

$$
\operatorname{tr}(DC)=15+6=21.
$$

Therefore

$$
\boxed{(CD)^T=D^TC^T=\begin{bmatrix}-2&-4\\9&23\end{bmatrix},\qquad
\operatorname{tr}(CD)=\operatorname{tr}(DC)=21.}
$$

#### E5. Gradient of a scalar function

**Problem.** Let

$$
f(x,y)=3x^2+2xy+y^2-4x.
$$

Compute $\nabla f$ and evaluate it at $(1,-2)$.

**Solution.**

The gradient is

$$
\nabla f(x,y)=\begin{bmatrix}\frac{\partial f}{\partial x}\\\frac{\partial f}{\partial y}\end{bmatrix}.
$$

Differentiate with respect to $x$:

$$
\frac{\partial f}{\partial x}=6x+2y+0-4=6x+2y-4.
$$

Differentiate with respect to $y$:

$$
\frac{\partial f}{\partial y}=0+2x+2y+0=2x+2y.
$$

Therefore

$$
\nabla f(x,y)=\begin{bmatrix}6x+2y-4\\2x+2y\end{bmatrix}.
$$

At $(1,-2)$,

$$
6(1)+2(-2)-4=6-4-4=-2,
$$

and

$$
2(1)+2(-2)=2-4=-2.
$$

So

$$
\boxed{\nabla f(x,y)=\begin{bmatrix}6x+2y-4\\2x+2y\end{bmatrix},\qquad
\nabla f(1,-2)=\begin{bmatrix}-2\\-2\end{bmatrix}.}
$$

### 🔴 Advanced

#### A1. Matrix-matrix multiplication and shape debugging

**Problem.** Let

$$
A=\begin{bmatrix}1&0&2\\-1&3&1\end{bmatrix},
\qquad
B=\begin{bmatrix}2&1\\1&-2\\0&4\end{bmatrix}.
$$

Compute $AB$, express it as outer products, and explain the shape of $BA$.

**Solution.**

Since $A\in\mathbb{R}^{2\times3}$ and $B\in\mathbb{R}^{3\times2}$, $AB$ is $2\times2$.

$$
AB=\begin{bmatrix}
1(2)+0(1)+2(0)&1(1)+0(-2)+2(4)\\
(-1)(2)+3(1)+1(0)&(-1)(1)+3(-2)+1(4)
\end{bmatrix}
=\begin{bmatrix}2&9\\1&-3\end{bmatrix}.
$$

For the outer-product form,

$$
a_{c,1}b_{r,1}^T=\begin{bmatrix}1\\-1\end{bmatrix}\begin{bmatrix}2&1\end{bmatrix}
=\begin{bmatrix}2&1\\-2&-1\end{bmatrix},
$$

$$
a_{c,2}b_{r,2}^T=\begin{bmatrix}0\\3\end{bmatrix}\begin{bmatrix}1&-2\end{bmatrix}
=\begin{bmatrix}0&0\\3&-6\end{bmatrix},
$$

$$
a_{c,3}b_{r,3}^T=\begin{bmatrix}2\\1\end{bmatrix}\begin{bmatrix}0&4\end{bmatrix}
=\begin{bmatrix}0&8\\0&4\end{bmatrix}.
$$

Adding gives

$$
\begin{bmatrix}2&1\\-2&-1\end{bmatrix}
+\begin{bmatrix}0&0\\3&-6\end{bmatrix}
+\begin{bmatrix}0&8\\0&4\end{bmatrix}
=\begin{bmatrix}2&9\\1&-3\end{bmatrix}.
$$

The reverse product has shape $3\times3$:

$$
BA=\begin{bmatrix}2&1\\1&-2\\0&4\end{bmatrix}\begin{bmatrix}1&0&2\\-1&3&1\end{bmatrix}
=\begin{bmatrix}1&3&5\\3&-6&0\\-4&12&4\end{bmatrix}.
$$

Therefore

$$
\boxed{AB=\begin{bmatrix}2&9\\1&-3\end{bmatrix},\qquad BA\in\mathbb{R}^{3\times3}\textrm{ and }BA\neq AB.}
$$

#### A2. Eigenvalues, eigenvectors, and diagonalization check

**Problem.** Let

$$
M=\begin{bmatrix}2&1\\1&2\end{bmatrix}.
$$

Find eigenvalues, normalized eigenvectors, and verify $M=U\Lambda U^T$.

**Solution.**

Compute

$$
M-\lambda I=\begin{bmatrix}2-\lambda&1\\1&2-\lambda\end{bmatrix}.
$$

The characteristic equation is

$$
\det(M-\lambda I)=(2-\lambda)^2-1=0.
$$

Expand and factor:

$$
(2-\lambda)^2-1=4-4\lambda+\lambda^2-1=\lambda^2-4\lambda+3=(\lambda-3)(\lambda-1).
$$

So $\lambda_1=3$ and $\lambda_2=1$.

For $\lambda_1=3$,

$$
M-3I=\begin{bmatrix}-1&1\\1&-1\end{bmatrix}.
$$

The equation $-z_1+z_2=0$ gives $z_2=z_1$, so

$$
z^{(1)}=\begin{bmatrix}1\\1\end{bmatrix},\qquad
u_1=\frac1{\sqrt2}\begin{bmatrix}1\\1\end{bmatrix}.
$$

For $\lambda_2=1$,

$$
M-I=\begin{bmatrix}1&1\\1&1\end{bmatrix}.
$$

The equation $z_1+z_2=0$ gives $z_2=-z_1$, so

$$
z^{(2)}=\begin{bmatrix}1\\-1\end{bmatrix},\qquad
u_2=\frac1{\sqrt2}\begin{bmatrix}1\\-1\end{bmatrix}.
$$

Set

$$
U=\frac1{\sqrt2}\begin{bmatrix}1&1\\1&-1\end{bmatrix},
\qquad
\Lambda=\begin{bmatrix}3&0\\0&1\end{bmatrix}.
$$

Then

$$
U\Lambda=\frac1{\sqrt2}\begin{bmatrix}3&1\\3&-1\end{bmatrix}.
$$

Now

$$
U\Lambda U^T
=\frac12\begin{bmatrix}3&1\\3&-1\end{bmatrix}\begin{bmatrix}1&1\\1&-1\end{bmatrix}
=\frac12\begin{bmatrix}4&2\\2&4\end{bmatrix}
=\begin{bmatrix}2&1\\1&2\end{bmatrix}=M.
$$

Therefore

$$
\boxed{M=U\Lambda U^T,\quad
U=\frac1{\sqrt2}\begin{bmatrix}1&1\\1&-1\end{bmatrix},\quad
\Lambda=\begin{bmatrix}3&0\\0&1\end{bmatrix}.}
$$

#### A3. PSD test through a quadratic form

**Problem.** Let

$$
Q=\begin{bmatrix}2&-1\\-1&2\end{bmatrix}.
$$

Derive $x^TQx$ and decide whether $Q\succeq0$ or $Q\succ0$.

**Solution.**

Let

$$
x=\begin{bmatrix}x_1\\x_2\end{bmatrix}.
$$

Because $Q^T=Q$, symmetry holds. Compute

$$
Qx=\begin{bmatrix}2x_1-x_2\\-x_1+2x_2\end{bmatrix}.
$$

Then

$$
x^TQx=x_1(2x_1-x_2)+x_2(-x_1+2x_2).
$$

Expand:

$$
x^TQx=2x_1^2-x_1x_2-x_1x_2+2x_2^2=2x_1^2-2x_1x_2+2x_2^2.
$$

Complete the square:

$$
2x_1^2-2x_1x_2+2x_2^2
=(x_1^2-2x_1x_2+x_2^2)+(x_1^2+x_2^2).
$$

Therefore

$$
x^TQx=(x_1-x_2)^2+x_1^2+x_2^2.
$$

Every term is nonnegative, so $x^TQx\geq0$ for all $x$. If $x\neq0$, then $x_1^2+x_2^2>0$, so $x^TQx>0$.

As an eigenvalue check,

$$
\det(Q-\lambda I)=\det\begin{bmatrix}2-\lambda&-1\\-1&2-\lambda\end{bmatrix}=(2-\lambda)^2-1.
$$

Setting this to zero gives

$$
(2-\lambda)^2=1
\quad\Rightarrow\quad
2-\lambda=\pm1,
$$

so $\lambda=1$ or $\lambda=3$, both positive.

$$
\boxed{x^TQx=(x_1-x_2)^2+x_1^2+x_2^2,\qquad Q\succ0.}
$$

#### A4. Hessian and convexity of a quadratic objective

**Problem.** Let

$$
g(x)=\frac12 x^T\begin{bmatrix}4&1\\1&3\end{bmatrix}x-\begin{bmatrix}1\\2\end{bmatrix}^Tx,
\qquad x=\begin{bmatrix}x_1\\x_2\end{bmatrix}.
$$

Derive the gradient, Hessian, convexity conclusion, and stationary point.

**Solution.**

Let

$$
H=\begin{bmatrix}4&1\\1&3\end{bmatrix},
\qquad b=\begin{bmatrix}1\\2\end{bmatrix}.
$$

Compute

$$
Hx=\begin{bmatrix}4x_1+x_2\\x_1+3x_2\end{bmatrix}.
$$

Then

$$
x^THx=x_1(4x_1+x_2)+x_2(x_1+3x_2)=4x_1^2+2x_1x_2+3x_2^2.
$$

So

$$
g(x_1,x_2)=2x_1^2+x_1x_2+\frac32x_2^2-x_1-2x_2.
$$

Differentiate:

$$
\frac{\partial g}{\partial x_1}=4x_1+x_2-1,
\qquad
\frac{\partial g}{\partial x_2}=x_1+3x_2-2.
$$

Thus

$$
\nabla g(x)=\begin{bmatrix}4x_1+x_2-1\\x_1+3x_2-2\end{bmatrix}=Hx-b.
$$

Take second derivatives:

$$
\frac{\partial^2 g}{\partial x_1^2}=4,\quad
\frac{\partial^2 g}{\partial x_1\partial x_2}=1,\quad
\frac{\partial^2 g}{\partial x_2\partial x_1}=1,\quad
\frac{\partial^2 g}{\partial x_2^2}=3.
$$

Therefore

$$
\nabla^2g(x)=\begin{bmatrix}4&1\\1&3\end{bmatrix}=H.
$$

The leading principal minors are

$$
4>0,\qquad \det(H)=4(3)-1(1)=11>0.
$$

So $H\succ0$, and $g$ is strictly convex. Set $\nabla g=0$:

$$
4x_1+x_2=1,\qquad x_1+3x_2=2.
$$

From the first equation, $x_2=1-4x_1$. Substitute:

$$
x_1+3(1-4x_1)=2
\quad\Rightarrow\quad
x_1+3-12x_1=2.
$$

Thus

$$
-11x_1=-1
\quad\Rightarrow\quad
x_1=\frac1{11}.
$$

Then

$$
x_2=1-4\left(\frac1{11}\right)=\frac{11}{11}-\frac4{11}=\frac7{11}.
$$

Therefore

$$
\boxed{\nabla g(x)=\begin{bmatrix}4x_1+x_2-1\\x_1+3x_2-2\end{bmatrix},\quad
\nabla^2g(x)=\begin{bmatrix}4&1\\1&3\end{bmatrix}\succ0,\quad
x^*=\begin{bmatrix}\frac1{11}\\\frac7{11}\end{bmatrix}.}
$$

#### A5. Matrix calculus with trace and determinant

**Problem.** For invertible $A$, let

$$
h(A)=\operatorname{tr}(ABA^TC)+\log|A|.
$$

Use

$$
\nabla_A\operatorname{tr}(ABA^TC)=CAB+C^TAB^T,
\qquad
\nabla_A\log|A|=(A^{-1})^T,
$$

and evaluate $\nabla_Ah(A)$ at

$$
A=\begin{bmatrix}1&2\\0&1\end{bmatrix},
\qquad
B=\begin{bmatrix}2&0\\1&3\end{bmatrix},
\qquad
C=\begin{bmatrix}1&-1\\2&0\end{bmatrix}.
$$

**Solution.**

All matrices are $2\times2$, so each gradient term must also be $2\times2$. By linearity,

$$
\nabla_Ah(A)=CAB+C^TAB^T+(A^{-1})^T.
$$

Compute $CAB$. First,

$$
CA=\begin{bmatrix}1&-1\\2&0\end{bmatrix}\begin{bmatrix}1&2\\0&1\end{bmatrix}
=\begin{bmatrix}1&1\\2&4\end{bmatrix}.
$$

Then

$$
CAB=\begin{bmatrix}1&1\\2&4\end{bmatrix}\begin{bmatrix}2&0\\1&3\end{bmatrix}
=\begin{bmatrix}1(2)+1(1)&1(0)+1(3)\\2(2)+4(1)&2(0)+4(3)\end{bmatrix}
=\begin{bmatrix}3&3\\8&12\end{bmatrix}.
$$

Next, since

$$
C^T=\begin{bmatrix}1&2\\-1&0\end{bmatrix},
\qquad
B^T=\begin{bmatrix}2&1\\0&3\end{bmatrix},
$$

we get

$$
C^TA=\begin{bmatrix}1&2\\-1&0\end{bmatrix}\begin{bmatrix}1&2\\0&1\end{bmatrix}
=\begin{bmatrix}1&4\\-1&-2\end{bmatrix}.
$$

Therefore

$$
C^TAB^T=\begin{bmatrix}1&4\\-1&-2\end{bmatrix}\begin{bmatrix}2&1\\0&3\end{bmatrix}
=\begin{bmatrix}2&13\\-2&-7\end{bmatrix}.
$$

For the inverse-transpose term,

$$
|A|=1(1)-2(0)=1,
$$

so

$$
A^{-1}=\begin{bmatrix}1&-2\\0&1\end{bmatrix},
\qquad
(A^{-1})^T=\begin{bmatrix}1&0\\-2&1\end{bmatrix}.
$$

Add the terms:

$$
\nabla_Ah(A)=\begin{bmatrix}3&3\\8&12\end{bmatrix}
+\begin{bmatrix}2&13\\-2&-7\end{bmatrix}
+\begin{bmatrix}1&0\\-2&1\end{bmatrix}.
$$

Entrywise,

$$
3+2+1=6,\qquad 3+13+0=16,
$$

$$
8-2-2=4,\qquad 12-7+1=6.
$$

Therefore

$$
\boxed{\nabla_Ah(A)=\begin{bmatrix}6&16\\4&6\end{bmatrix}.}
$$